---
title: "05. Online serving & promotion"
description: "A FastAPI serving container that loads one exact registry version at startup, with promotion and rollback as an alias flip plus a container recreate: never a rebuild."
categories: []
---

## Outcome

When a model needs low-latency online inference, a long-running **serving container** loads an **exact MLflow model version** at startup and exposes an HTTP prediction endpoint. The image carries code only, so changing the live model is never a rebuild: **promotion** flips the registry's `production` alias to version N and recreates the one long-running consumer pinned to it, and **rollback** repeats that operation at the previous version number. This chapter builds the serving branch on the Part I Compose stack from [chapter 02](02-local-foundation.ipynb); [Part II](10-azure-foundation.ipynb) runs the same image as a Serving ACA App ([chapter 11](11-porting-to-aca.ipynb)). Online serving is optional: batch-only deployments skip it entirely.


## Design — one pinned version, one redeploy path

Three decisions shape the design:

- **Own image.** `src/serving_app/` builds a small FastAPI image (`python:3.11-slim` base; mlflow/sklearn/fastapi/uvicorn pinned) that loads `models:/<name>/<version>` from the self-hosted registry at startup. Framework, runtime, and dependencies stay under project control.
- **Exact version pinning.** The served version arrives as the `MODEL_VERSION` environment variable and must name one specific registry version: never `latest`, never a floating alias resolved per request. The container refuses to start without it, so "some recent version" is not a state the platform can be in.
- **Verifiability.** The resolved version is logged at startup and reported by `/readyz`, so the live version is an observable fact available to probes and clients alike.

Promotion and rollback then fall out as consequences, stated as five rules that [chapter 08](08-environment-contract.ipynb) adopts into the environment contract:

1. **Images carry code only.** Changing the live model never rebuilds an image.
2. **Models are registry versions.** An evaluated version becomes live through one atomic alias flip, `production → N`, against the same registry in both parts of the course.
3. **Only the long-running consumer is redeployed.** Serving is the only component holding a model in memory, loaded once at container startup, so promotion repins its `MODEL_VERSION` and recreates it.
4. **Ephemeral jobs need nothing.** The next batch execution receives the version as an ordinary parameter.
5. **Consumers never poll.** There is no hot reload; the live version is exactly what `/readyz` reports.

**Rollback** is rules 2 and 3 replayed at the previous version number: flip the alias back, recreate the container. Nothing is rebuilt on either path, because versions are pointers into the registry rather than artifacts to reproduce. Provenance stays readable by hand: registered version, its evaluation record, Git tag, deployed image digest.


## Build in `projects/ml-platform/`

```
projects/ml-platform/
├── demo/
│   ├── docker-compose.yml      # serving service: MODEL_VERSION=${DEMO_MODEL_VERSION:-1}, :18080
│   ├── promote.py              # promotion entrypoint: alias flip + consumer redeploy, two backends
│   └── golden_path.py          # end-to-end suite: train → promote → /readyz assertion → batch
└── src/serving_app/
    ├── Dockerfile              # python:3.11-slim + FastAPI + MLflow; HEALTHCHECK curls /healthz
    ├── requirements.txt        # mlflow pinned to the same version as the registry server
    └── app.py                  # loads models:/name/N once at startup → /healthz, /readyz,
                                #   /v1/predictions
```

`serving` is one of the stack's two long-lived services ([chapter 02](02-local-foundation.ipynb) tours the rest). It builds from `src/serving_app/Dockerfile`, publishes `localhost:18080`, and waits for `train` to complete successfully and `mlflow` to turn healthy, because the pinned version must exist before startup can load it. Its entire model configuration is two variables: `MODEL_NAME=wine-quality` and `MODEL_VERSION=${DEMO_MODEL_VERSION:-1}`. The Compose interpolation is the pin point: a fresh stack serves version 1, and `demo/.env`, written by every promotion below, overrides it thereafter.

:::{.callout-note}
## Part II preview

[Chapter 11](11-porting-to-aca.ipynb) declares this same container as a Serving ACA App from `infra/modules/serving_app/`: managed identity (`id-serving`) replaces plaintext registry access, ACA readiness/liveness probes point at the same `/healthz` and `/readyz`, and an ingress FQDN replaces the published port. The image itself is reused unchanged; only its configuration differs.
:::

## How the pieces connect

### Startup: load once, canary, then ready

app.py registers a FastAPI lifespan hook, so the model loads once at container
startup, not per request. Startup is all-or-nothing:

1. Read MLFLOW_TRACKING_URI, MODEL_NAME, and MODEL_VERSION from the environment.
   With MODEL_VERSION unset, the App records the error and refuses to serve:
   no floating aliases.
2. Call mlflow.pyfunc.load_model("models:/<name>/<version>"). This is the common
   loading interface for both the tabular sklearn artifact and the text-column
   LLM artifact.
3. Run a canary prediction. Tabular models receive the fixed feature vector;
   text models receive a small input record. This proves the loaded artifact is
   callable before the readiness flag flips.

If any step raises, the process stays alive but never reports ready, and the
reason is visible on /readyz.

### Probes

/healthz is liveness: always 200 once the Python process runs (the Dockerfile
HEALTHCHECK curls it). /readyz is readiness: 503 with the failure reason until
the canary passes, then 200 carrying {status, model_name, model_version}. Only
the ready state carries a version, so a green /readyz doubles as proof of which
model is live.

### Inference

POST /v1/predictions accepts tabular instances such as
{"instances": [[f1, f2, ...]]}; text models also accept string instances or
{"input": "..."} records. The response includes predictions, model_name, and
model_version. Every response self-identifies, so a client can tell which
version served it without probing. Requests arriving before readiness draw a
503; an empty instances list draws a 422.


## Promotion and rollback in practice

One entrypoint performs both halves of a promotion in the right order. `demo/promote.py` first makes the atomic registry change, setting the `production` alias to N against `MLFLOW_TRACKING_URI`, then redeploys the consumer the way its environment expects. The `--backend` flag selects the redeploy:

| Backend | Consumer redeploy | Part |
|---|---|---|
| `local` (default) | merges `DEMO_MODEL_VERSION=N` into `demo/.env`, preserving unrelated lines, then runs `docker compose up -d serving`, recreating only the serving service | I |
| `aca` | prints `az containerapp update --set-env-vars MODEL_VERSION=N`, which rolls a new serving App revision; applied only with `--execute` | II |

Both branches live in one script from day one so the promotion *contract* is identical across parts; the `aca` branch defaults to a dry run because no Azure environment exists until Part II.

**Rollback needs no separate tooling**: it is a previous promotion replayed. After a bad version 3, promoting version 2 again flips the alias back and recreates the container on version 2. On Azure the same command rolls a new revision pinned at the older value; [chapter 13](13-azure-operations.ipynb) treats promotion and rollback generally as definition updates rather than rebuilds, and [chapter 12](12-ci-cd.ipynb) moves those definition updates into CI.


```bash
# Promote version 3 locally: alias flip, then only the serving container restarts.
python demo/promote.py --backend local --version 3

# The same promotion against Azure: alias flip, then a new serving revision.
# Prints the az command by default; add --execute to apply it.
python demo/promote.py --backend aca --version 3 \
    --resource-group rg-mlp --app-name app-serving

# Rollback: the identical operation aimed at the previous version.
python demo/promote.py --backend local --version 2
```


The command returning means the redeploy has been issued, not that the new version is being served: Compose still has to stop the old container, start the new one, load the model, and clear the canary. Promotion is complete only when GET /readyz answers {"status": "ready", "model_name": ..., "model_version": N}. That exact-version property turns "which model is live?" from a question about deployment history into an observable fact.

It is also what makes the chapter mechanically testable. demo/golden_path.py drives the full path over plain HTTP against a running Compose stack. The Azure smoke adapters in deploy/ use Azure's Job and App control-plane APIs separately, then assert the same terminal status, results, readiness, model identity, and prediction behavior.


```bash
# From projects/ml-platform/demo/, with the stack up:
python demo/golden_path.py
```


Each step asserts before the next begins:

1. **Train:** trigger training through the dashboard API and poll the results row to `SUCCESS`.
2. **Resolve:** read the newest registered version N from the MLflow REST API.
3. **Promote:** invoke `demo/promote.py` in a subprocess, the real entrypoint rather than a reimplementation.
4. **Serve:** poll `/readyz` until `status` is `ready`, then assert `model_name` and `model_version` equal the promoted values exactly; a stale version fails the suite.
5. **Score:** trigger batch scoring pinned to N and poll to terminal status.
6. **Record:** assert the batch parent row shows `SUCCESS` in the results API.

**Acceptance evidence** is the suite printing `GOLDEN PATH: PASS`: promotion lands the exact promoted version behind `/readyz`, rollback replays an older promotion cleanly, and every prediction response echoes the version that produced it.

## Extensions (deferred from the MVP)

| Deferred | Contract | MVP substitute |
|---|---|---|
| Token/scope auth on the prediction endpoint | `docs/05` | Trust the network boundary (Compose network now, ACA ingress later) |
| Autoscaling on HTTP concurrency | `docs/05` | Single warm replica |
| LLM shared-budget partitioning | `docs/05` | N/A (classical model) |
| Automated four-part release provenance | `docs/06` | Git tag plus recorded digest, by hand |

Next: **[06 — Observability & dashboard](06-observability-and-dashboard.ipynb)**
makes the running platform visible and launchable.
